# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pinkk1808/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: One row will represent one pseudonymized content page within a client after aggregating the daily warehouse data. The raw fact_content_daily_performance table is daily data, but for my Refresh / Content Opportunity Scoring lane I will create page-level rows for ranking.

Feature window: March 1–31, 2026.
Outcome window: April 1–30, 2026.

I will use March search-performance signals to rank pages by their risk of future decline, then use April performance as the later outcome. This keeps the information used for ranking earlier than the outcome I am trying to evaluate.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%pip -q install duckdb huggingface_hub

from google.colab import userdata
import duckdb

# Read the token safely from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

assert HF_TOKEN and HF_TOKEN.startswith("hf_"), \
    "HF_TOKEN secret পাওয়া যায়নি. Secrets panel-e notebook access ON ache kina check koro."

# Connect DuckDB to the gated Hugging Face warehouse
con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

# We will work with a mid-panel month, not the final-month sample
MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
APR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"

DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"

print("Connected to FlyRank warehouse ✅")
print("Feature window: March 2026")
print("Outcome window: April 2026")

Connected to FlyRank warehouse ✅
Feature window: March 2026
Outcome window: April 2026


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature fields:

gsc_impressions — March search visibility, known before the April outcome.
gsc_clicks — March search clicks, known before the April outcome.
gsc_avg_position — March average search position, known before the April outcome.
March CTR derived from gsc_clicks / gsc_impressions — known before the April outcome.
Measured GSC days in March — derived from gsc_data_available; this tells me how many days had usable search data before the April outcome.

Label / proxy:
My proxy label will represent whether a page declines in the later April outcome window compared with its March performance. The April values used to define this outcome are label information and must not be used as model features.

Context fields:
client_hash_id and content_hash_id are used only for grouping, joining, and later client-aware validation. They are not model features.

Excluded fields:

Any April/future performance columns — excluded because they contain information from the outcome window and would cause leakage.
Query-table fields whose fixed 90-day window overlaps the outcome period — excluded unless their time window is proven safe.
Client/content IDs as predictive features — excluded because they are identifiers, not meaningful page signals.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [7]:
# Query 1 — verify the grain
grain_check = con.sql(f"""
SELECT
    COUNT(*) AS unique_page_day_groups,
    SUM(CASE WHEN n > 1 THEN 1 ELSE 0 END) AS duplicated_groups
FROM (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS n
    FROM {MAR}
    GROUP BY 1, 2, 3
)
""").df()

print("QUERY 1 — Grain check")
display(grain_check)


# Query 2 — row count and date span
slice_check = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {MAR}
""").df()

print("QUERY 2 — March slice size and date span")
display(slice_check)


# Query 3 — availability check
availability_check = con.sql(f"""
SELECT
    COUNT(*) AS available_rows
FROM {MAR}
WHERE gsc_data_available IS TRUE
""").df()

print("QUERY 3 — Rows surviving gsc_data_available IS TRUE")
display(availability_check)

QUERY 1 — Grain check


,unique_page_day_groups,duplicated_groups
0,9841378,0.0


QUERY 2 — March slice size and date span


,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


QUERY 3 — Rows surviving gsc_data_available IS TRUE


,available_rows
0,3611061


In [8]:
feature_frame = con.sql(f"""
WITH mar AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN gsc_impressions ELSE 0 END) AS imp_mar,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN gsc_clicks ELSE 0 END) AS clk_mar,
        AVG(CASE WHEN gsc_data_available IS TRUE THEN gsc_avg_position END) AS pos_mar,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS measured_days_mar
    FROM {MAR}
    GROUP BY 1, 2
),
apr AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN gsc_impressions ELSE 0 END) AS imp_apr,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS measured_days_apr
    FROM {APR}
    GROUP BY 1, 2
)
SELECT
    m.client_hash_id,
    m.content_hash_id,

    m.imp_mar * 1.0 / NULLIF(m.measured_days_mar, 0) AS imp_per_day_mar,
    m.clk_mar * 1.0 / NULLIF(m.measured_days_mar, 0) AS clicks_per_day_mar,
    m.clk_mar * 1.0 / NULLIF(m.imp_mar, 0) AS ctr_mar,
    m.pos_mar,
    m.measured_days_mar,

    a.imp_apr * 1.0 / NULLIF(a.measured_days_apr, 0) AS imp_per_day_apr

FROM mar m
JOIN apr a
  USING (client_hash_id, content_hash_id)

WHERE
    m.measured_days_mar > 0
    AND a.measured_days_apr > 0
    AND m.imp_mar >= 100
""").df()

# Future outcome / proxy label
feature_frame["declined_next_month"] = (
    feature_frame["imp_per_day_apr"]
    < 0.8 * feature_frame["imp_per_day_mar"]
).astype(int)

five_features = [
    "imp_per_day_mar",
    "clicks_per_day_mar",
    "ctr_mar",
    "pos_mar",
    "measured_days_mar"
]

print("Rows in feature frame:", len(feature_frame))
print("Five honest features:", five_features)
print(
    "Declining next month:",
    feature_frame["declined_next_month"].sum()
)

display(
    feature_frame[
        five_features + ["declined_next_month"]
    ].head()
)

Rows in feature frame: 100893
Five honest features: ['imp_per_day_mar', 'clicks_per_day_mar', 'ctr_mar', 'pos_mar', 'measured_days_mar']
Declining next month: 52057


,imp_per_day_mar,clicks_per_day_mar,ctr_mar,pos_mar,measured_days_mar,declined_next_month
0,6.241379,0.000000,0.000000,5.147402,29.0,1
1,29.000000,0.032258,0.001112,5.145765,31.0,0
2,103.600000,0.000000,0.000000,6.969536,30.0,1
3,10.612903,0.000000,0.000000,5.177774,31.0,1
4,24.903226,0.032258,0.001295,4.685335,31.0,1


Feature availability:

imp_per_day_mar — knowable at the decision moment because it uses only March impressions.
clicks_per_day_mar — knowable because it uses only March clicks.
ctr_mar — knowable because it is calculated only from March clicks and impressions.
pos_mar — knowable because it uses only March average search position.
measured_days_mar — knowable because it only counts March days where GSC data was available.

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

# Keep only rows with usable honest features
model_df = feature_frame.dropna(
    subset=five_features + ["declined_next_month"]
).copy()

# Same train/test rows for both comparisons
train_idx, test_idx = train_test_split(
    model_df.index,
    test_size=0.25,
    random_state=42,
    stratify=model_df["declined_next_month"]
)

# Honest model: March-only features
honest_model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

honest_model.fit(
    model_df.loc[train_idx, five_features],
    model_df.loc[train_idx, "declined_next_month"]
)

honest_score = roc_auc_score(
    model_df.loc[test_idx, "declined_next_month"],
    honest_model.predict_proba(
        model_df.loc[test_idx, five_features]
    )[:, 1]
)

# DELIBERATE LEAK:
# This uses April outcome information, so it must never be a real feature.
model_df["leak_apr_vs_mar_ratio"] = (
    model_df["imp_per_day_apr"] /
    model_df["imp_per_day_mar"]
)

leaky_features = five_features + ["leak_apr_vs_mar_ratio"]

leaky_model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

leaky_model.fit(
    model_df.loc[train_idx, leaky_features],
    model_df.loc[train_idx, "declined_next_month"]
)

leaky_score = roc_auc_score(
    model_df.loc[test_idx, "declined_next_month"],
    leaky_model.predict_proba(
        model_df.loc[test_idx, leaky_features]
    )[:, 1]
)

print("Honest ROC-AUC:", round(honest_score, 3))
print("Leaky ROC-AUC:", round(leaky_score, 3))

# Remove the illegal future-derived feature
model_df.drop(columns=["leak_apr_vs_mar_ratio"], inplace=True)

print(
    "Leak removed:",
    "leak_apr_vs_mar_ratio" not in model_df.columns
)

Honest ROC-AUC: 0.621
Leaky ROC-AUC: 1.0
Leak removed: True


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Data limitation: This slice only includes pages with usable GSC data, and my feature frame also keeps pages with at least 100 March impressions. That means the results are more representative of pages that already have measurable search visibility and may not generalize well to pages with very little or no visibility. Also, this experiment uses only March as the feature window and April as the outcome window, so the observed relationship may not stay the same across other months. Finally, the decline label is only a proxy based on lower impressions per day; it does not prove that a content refresh would cause performance to improve.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.